In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

X_train, y_train = train_df["clean_text"], train_df["sentiment"]
X_test, y_test = test_df["clean_text"], test_df["sentiment"]
labels = ["negative", "neutral", "positive"]
print(len(X_train), len(X_test))

27700 6925


In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
 

dummy = DummyClassifier(strategy="most_frequent", random_state=42)
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)
 
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.3f} | Macro F1: {f1:.3f}") 
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.933 | Macro F1: 0.322
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00       162
     neutral       0.00      0.00      0.00       300
    positive       0.93      1.00      0.97      6463

    accuracy                           0.93      6925
   macro avg       0.31      0.33      0.32      6925
weighted avg       0.87      0.93      0.90      6925



In [16]:
import re
from nltk.classify import NaiveBayesClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

results = []

# ---------- 1. Naive Bayes (NLTK) ----------
# NLTK needs each review as a dictionary {word: True}
def to_features(text):
    return {word: True for word in re.findall(r"[a-z']+", text)}

train_set = [(to_features(text), label) for text, label in zip(X_train, y_train)]
nb = NaiveBayesClassifier.train(train_set)

pred = [nb.classify(to_features(text)) for text in X_test]
results.append({"model": "Naive Bayes (NLTK)",
                "accuracy": accuracy_score(y_test, pred),
                "macro_f1": f1_score(y_test, pred, average="macro")})

nb.show_most_informative_features(10)   # words that matter most

# ---------- 2-3. SVM and Logistic Regression (TF-IDF) ----------
models = {
    "SVM": LinearSVC(class_weight="balanced"),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
}

for name, classifier in models.items():
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)), ("clf", classifier)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({"model": name,
                    "accuracy": accuracy_score(y_test, pred),
                    "macro_f1": f1_score(y_test, pred, average="macro")})

pd.DataFrame(results).round(3)

Most Informative Features
                   avail = True           negati : positi =     92.7 : 1.0
                    nope = True           negati : positi =     92.7 : 1.0
                tomorrow = True           negati : positi =     92.7 : 1.0
                   worst = True           negati : positi =     90.3 : 1.0
                    logo = True           negati : positi =     87.4 : 1.0
             refurbished = True           negati : positi =     87.4 : 1.0
                 refused = True           negati : positi =     87.4 : 1.0
                mediocre = True           neutra : positi =     81.9 : 1.0
                  faulty = True           negati : positi =     71.5 : 1.0
               returning = True           negati : positi =     70.5 : 1.0


,model,accuracy,macro_f1
0,Naive Bayes (NLTK),0.579,0.341
1,SVM,0.942,0.602
2,Logistic Regression,0.901,0.615


In [17]:
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
model.fit(X_train, y_train)
test_pred = model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, test_pred, digits=3))

              precision    recall  f1-score   support

    negative      0.478     0.667     0.557       162
     neutral      0.253     0.500     0.336       300
    positive      0.979     0.925     0.952      6463

    accuracy                          0.901      6925
   macro avg      0.570     0.697     0.615      6925
weighted avg      0.936     0.901     0.916      6925

